# 08 · Bootstrap confidence intervals (Chloe)

Every number in the README's *Key results* is a point estimate from one sample of applicants. This notebook puts 95% confidence intervals on those numbers, and on each fix's change vs baseline, so we can tell which differences are real and which could be sampling noise.


## 0 · Setup

**What this does**
- Fixes the agreed settings: seed 42, 1000 resamples, 95% percentile interval (2.5th–97.5th percentile).

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/CS3268_Project'
except ImportError:                      # local test run only
    import os; ROOT = os.environ['CS3268_ROOT']
PREDS, MODELS_DIR = f'{ROOT}/preds', f'{ROOT}/models'

import numpy as np, pandas as pd
SEED, N_BOOT = 42, 1000                  # agreed: percentile CIs, seed 42, 1000 resamples
LO_Q, HI_Q = 2.5, 97.5                   # 95% percentile interval

Mounted at /content/drive


## 1 · Load the five prediction files

**What this does**
- Reads one file per model. Each has columns `id, month, y_true, age_group, score, decision`.
- Prints row counts and months, so you can see every file covers months 5–7.
- Uses **only** the five agreed files. `preds/` also has `preds_expgrad_db*.parquet` from the 23 Oct sweep; those are ignored.
- **Never touches `score`.** Only the saved `decision` column is used later, so each model's cutoff stays fixed. We are measuring the system as built, not re-tuning it.

**From the notes: L9 slides 4 and 6** (where you can intervene in the pipeline). Each model is a different stage:

| label | file | stage (L9 s4/s6) |
|---|---|---|
| baseline | `preds_baseline.parquet` | the model being fixed |
| reweigh | `preds_reweigh.parquet` | pre-processing: reweight training data (L9 s9–11, W = P_exp / P_obs) |
| expgrad_0.05 | `preds_expgrad.parquet` | in-processing: fairness constraint during training (pre-registered winner) |
| expgrad_0.01 | `preds_expgrad_prelim_0923.parquet` | in-processing (overcorrected version) |
| groupthr | `preds_groupthr.parquet` | post-processing: separate cutoff per age group |

In [ ]:
FILES = {
    'baseline':     'preds_baseline.parquet',
    'reweigh':      'preds_reweigh.parquet',
    'groupthr':     'preds_groupthr.parquet',
    'expgrad_0.05': 'preds_expgrad.parquet',
    'expgrad_0.01': 'preds_expgrad_prelim_0923.parquet',
}
MODEL_NAMES = list(FILES)
SPLIT_MONTHS = {'val': [5], 'test': [6, 7]}   # common.SPLITS

raw = {m: pd.read_parquet(f'{PREDS}/{f}') for m, f in FILES.items()}
for m, d in raw.items():
    print(f'{m:13s} rows={len(d):,}  months={sorted(map(int, d.month.unique()))}  cols={list(d.columns)}')

baseline      rows=324,334  months=[5, 6, 7]  cols=['id', 'month', 'y_true', 'age_group', 'score', 'decision']
reweigh       rows=324,334  months=[5, 6, 7]  cols=['id', 'month', 'y_true', 'age_group', 'score', 'decision']
groupthr      rows=324,334  months=[5, 6, 7]  cols=['id', 'month', 'y_true', 'age_group', 'score', 'decision']
expgrad_0.05  rows=324,334  months=[5, 6, 7]  cols=['id', 'month', 'y_true', 'age_group', 'score', 'decision']
expgrad_0.01  rows=324,334  months=[5, 6, 7]  cols=['id', 'month', 'y_true', 'age_group', 'score', 'decision']


## 2 · Split into val / test and line up the rows

**What this does**
- Splits each file into **val** (month 5) and **test** (months 6–7).
- Sorts each file by `id`, so row *i* is the **same applicant in all five files**.
- Asserts that ids, true labels and age groups match the baseline exactly, and that every value is 0/1. If any assert fails, the files don't describe the same applicants, and the paired comparison in step 4 would be meaningless.
- Prints n, the 50-plus share and the fraud rate. They should match the README: val n = 119,323, test n = 205,011.

**Why line up rows:** step 4 uses a **paired** bootstrap. The same resampled applicants go into baseline and each fix, so the noise from which applicants got drawn cancels out of the before/after difference. Pairing is impossible unless the rows correspond.

**Why val and test separately — from the notes: L6 slide 14** (temporal drift; fraud is the slide's example) **and L7 slides 10–12** (covariate shift / label shift). The cutoff targets 5% FPR on month 5, but test FPR is 6.22%. Val-only results would overstate how well each fix holds up on new months.

In [ ]:
def align(split):
    out = {}
    for m, d in raw.items():
        s = d[d.month.isin(SPLIT_MONTHS[split])].sort_values('id').reset_index(drop=True)
        out[m] = s
    ref = out['baseline']
    assert ref.id.is_unique, 'duplicate ids'
    for m, s in out.items():
        assert len(s) == len(ref), f'{m}: {len(s)} rows vs baseline {len(ref)}'
        assert (s.id.values == ref.id.values).all(), f'{m}: ids differ from baseline'
        assert (s.y_true.values == ref.y_true.values).all(), f'{m}: labels differ'
        assert (s.age_group.values == ref.age_group.values).all(), f'{m}: age_group differs'
        for c in ('y_true', 'age_group', 'decision'):
            assert set(np.unique(s[c])) <= {0, 1}, f'{m}.{c} not 0/1'
    return out

aligned = {sp: align(sp) for sp in SPLIT_MONTHS}
for sp, a in aligned.items():
    b = a['baseline']
    print(f'{sp}: n={len(b):,}  50-plus share={b.age_group.mean():.3f}  fraud rate={b.y_true.mean():.4f}')
# README: val n=119,323; test n=205,011

val: n=119,323  50-plus share=0.174  fraud rate=0.0118
test: n=205,011  50-plus share=0.165  fraud rate=0.0140


## 3 · Define the fairness metrics

**What this does**
- Defines a function that turns **8 counts** into every metric we report. The 8 counts are age group (0 = under 50, 1 = 50-plus) × true label × decision, i.e. a confusion matrix per age group.
- Everything is computed from counts, so 1000 resamples × 5 models runs in seconds instead of looping over rows.
- Nothing is computed yet. This cell only defines the function used in steps 4–6.

**From the notes: L8 slide 22** builds the confusion matrix per group and defines FPR, TPR, PPV and selection rate. **L8 slide 23** defines *equalized odds* (equal FPR and TPR across groups) and *equal opportunity* (equal TPR).

| metric | formula | plain meaning | notes link |
|---|---|---|---|
| `fpr_u50`, `fpr_50p` | FP / (FP + TN) | share of **honest** applicants wrongly rejected | L8 s22; FPR half of equalized odds (s23) |
| `fpr_ratio` | min(FPR) / max(FPR) | 1.0 = parity; baseline 0.32 ≈ "3x" | equalized odds (s23) as one number |
| `fpr_overall` | pooled FP / (FP + TN) | sanity: val ≈ 5% by construction | L8 s9: the overall number hides the per-group gap |
| `recall_u50`, `recall_50p` | TP / (TP + FN) | share of fraud caught | TPR; equal opportunity (s23) |
| `recall_overall` | pooled TP / (TP + FN) | the **cost** of each fix | L8 s30 trade-off |
| `share_50p_of_wrongly_rejected_honest` | FP₅₀₊ / all FP | the README's "~39%" | L8 s9 |
| `share_50p_of_honest` | honest₅₀₊ / all honest | comparison point for the line above | |
| `precision_*`, `flag_rate_*` | TP/(TP+FP), flagged / n | bonus for group thresholds | L8 s22 (PPV, selection rate); s30 |

**Why min/max for the ratio:** ExpGrad 0.01 overcorrects, so under-50s become the more-flagged group. Dividing the smaller FPR by the larger keeps the ratio between 0 and 1 regardless of which group is worse off. It is recomputed inside each resample.

In [ ]:
def metrics_from_counts(c):
    # c[..., 8] counts in order (group, y_true, decision); returns dict of arrays shaped c.shape[:-1]
    c = c.reshape(c.shape[:-1] + (2, 2, 2)).astype(float)       # [..., group, y, d]
    TN, FP = c[..., 0, 0], c[..., 0, 1]                           # each [..., group]
    FN, TP = c[..., 1, 0], c[..., 1, 1]
    with np.errstate(divide='ignore', invalid='ignore'):
        fpr  = FP / (FP + TN)
        rec  = TP / (TP + FN)
        prec = TP / (TP + FP)
        flag = (TP + FP) / (TP + FP + TN + FN)
        return {
            'fpr_u50': fpr[..., 0], 'fpr_50p': fpr[..., 1],
            'fpr_ratio': np.minimum(fpr[..., 0], fpr[..., 1]) / np.maximum(fpr[..., 0], fpr[..., 1]),
            'fpr_overall': FP.sum(-1) / (FP + TN).sum(-1),
            'recall_u50': rec[..., 0], 'recall_50p': rec[..., 1],
            'recall_overall': TP.sum(-1) / (TP + FN).sum(-1),
            'share_50p_of_wrongly_rejected_honest': FP[..., 1] / FP.sum(-1),
            'share_50p_of_honest': (FP + TN)[..., 1] / (FP + TN).sum(-1),
            'precision_u50': prec[..., 0], 'precision_50p': prec[..., 1],
            'flag_rate_u50': flag[..., 0], 'flag_rate_50p': flag[..., 1],
        }

## 4 · Run the paired bootstrap

**What this does**, for val and then test:
1. Gives every row of every model a cell code 0–7 (`age_group*4 + y_true*2 + decision`).
2. Counts the cells on the full data. These are the **point estimates**.
3. 1000 times: draws `n` row numbers with replacement and counts the cells for **all five models using those same row numbers**. That shared draw is the pairing.
4. Stores all 1000 × 5 sets of counts. The next cell turns them into CIs.

Each split starts from a fresh `default_rng(42)`, so val and test are each reproducible on their own. It takes roughly 10–20 seconds.

**Why resample at all:** the 119k val applicants are one sample. A different month of applicants would give slightly different numbers. Resampling rows imitates that variation, and the spread of the 1000 results is the uncertainty.

**From the notes: L7 slide 9** says data dependence leads to "underestimated variance and overconfidence", which is the problem CIs address. **The bootstrap method itself is not in the course notes.** It is standard statistics (Efron's percentile bootstrap), so cite a statistics source in the report.

In [ ]:
def bootstrap(split):
    a = aligned[split]
    M, n = len(MODEL_NAMES), len(a['baseline'])
    codes = np.stack([a[m].age_group.values * 4 + a[m].y_true.values * 2 + a[m].decision.values
                      for m in MODEL_NAMES]).astype(np.int64)
    codes += (np.arange(M) * 8)[:, None]                          # give each model its own 8 bins
    full = np.bincount(codes.ravel(), minlength=8 * M).reshape(M, 8)
    rng = np.random.default_rng(SEED)
    boot = np.empty((N_BOOT, M, 8), dtype=np.int64)
    for r in range(N_BOOT):
        idx = rng.integers(0, n, size=n)                          # SAME rows for every model = paired
        boot[r] = np.bincount(codes[:, idx].ravel(), minlength=8 * M).reshape(M, 8)
    return full, boot

results = {}
for sp in SPLIT_MONTHS:
    results[sp] = bootstrap(sp)
    print(sp, 'done')

val done
test done


### 4b · Turn the resamples into the CI table

**What this does**
- For every model × split × metric: `estimate` is the value on the full data, and `lo` / `hi` are the 2.5th and 97.5th percentiles of the 1000 resampled values.
- **Change vs baseline** (for `fpr_ratio`, `recall_overall`, and per group `recall_u50`, `recall_50p`, `fpr_u50`, `fpr_50p`): within each resample it computes *fix minus baseline* on the same rows, then takes percentiles of those 1000 differences. That gives the `delta_*_vs_baseline` rows.
- **Why per group too:** the README makes per-group claims such as "ExpGrad lowers 50-plus recall 0.72 → 0.67". Two separate CIs that overlap don't settle whether a change is real. A paired CI on the difference does, because the same applicants are on both sides.
- Prints a warning if any resample produced an undefined value (e.g. a group with zero fraud cases drawn). With these sample sizes this should not happen.
- The result, `ci`, is the long table that gets saved: `model, split, metric, estimate, lo, hi`.

**From the notes: L8 slide 30** ("you can't have it all"). Each fix trades recall for fairness, and the delta rows put a CI on how much was gained and how much it cost.

In [ ]:
rows = []
b = MODEL_NAMES.index('baseline')
for sp, (full, boot) in results.items():
    pt, bs = metrics_from_counts(full), metrics_from_counts(boot)   # pt[k]: (M,), bs[k]: (N_BOOT, M)
    for k in pt:
        for j, m in enumerate(MODEL_NAMES):
            lo, hi = np.nanpercentile(bs[k][:, j], [LO_Q, HI_Q])
            rows.append(dict(model=m, split=sp, metric=k, estimate=pt[k][j], lo=lo, hi=hi))
    # Paired change vs baseline: difference taken INSIDE each resample (same rows), then percentiles.
    for k in ('fpr_ratio', 'recall_overall', 'recall_u50', 'recall_50p', 'fpr_u50', 'fpr_50p'):
        for j, m in enumerate(MODEL_NAMES):
            if j == b:
                continue
            d = bs[k][:, j] - bs[k][:, b]
            lo, hi = np.nanpercentile(d, [LO_Q, HI_Q])
            rows.append(dict(model=m, split=sp, metric=f'delta_{k}_vs_baseline',
                             estimate=pt[k][j] - pt[k][b], lo=lo, hi=hi))
    nan_frac = {k: np.isnan(v).mean() for k, v in bs.items() if np.isnan(v).any()}
    if nan_frac:
        print(sp, 'WARNING NaNs in resamples:', nan_frac)

ci = pd.DataFrame(rows)
print(len(ci), 'rows')
ci.head()

178 rows


,model,split,metric,estimate,lo,hi
0,baseline,val,fpr_u50,0.036843,0.035713,0.038042
1,reweigh,val,fpr_u50,0.050217,0.048764,0.051578
2,groupthr,val,fpr_u50,0.050002,0.048571,0.051328
3,expgrad_0.05,val,fpr_u50,0.042240,0.040953,0.043447
4,expgrad_0.01,val,fpr_u50,0.058644,0.057258,0.060011


## 5 · Sanity check against the README

**What this does**
- Compares the point estimates (full data, no resampling) with the numbers already in the README, and prints `ok` or `CHECK` for each.
- The point estimates don't depend on the bootstrap, so they **must** match. A `CHECK` means the wrong file was loaded, the split is wrong, or the README is stale.
- **Stop here if anything says `CHECK`, and sort it out before using any CI.**

**From the notes:** nothing specific. This is a reproducibility check (README: "every number must match a `preds/` file").

In [ ]:
EXPECTED = [  # (model, split, metric, README value, tolerance)
    ('baseline', 'val', 'fpr_overall', 0.0500, 0.001), ('baseline', 'test', 'fpr_overall', 0.0622, 0.001),
    ('baseline', 'val', 'fpr_u50', 0.0368, 0.001),     ('baseline', 'val', 'fpr_50p', 0.1135, 0.001),
    ('baseline', 'val', 'recall_overall', 0.5478, 0.001), ('baseline', 'test', 'recall_overall', 0.5893, 0.001),
    ('baseline', 'val', 'fpr_ratio', 0.32, 0.006),     ('baseline', 'test', 'fpr_ratio', 0.33, 0.006),
    ('baseline', 'val', 'share_50p_of_wrongly_rejected_honest', 0.39, 0.02),
    ('expgrad_0.05', 'val', 'fpr_ratio', 0.48, 0.006), ('expgrad_0.05', 'test', 'fpr_ratio', 0.47, 0.006),
    ('expgrad_0.05', 'val', 'recall_overall', 0.5485, 0.001),
    ('expgrad_0.01', 'val', 'fpr_ratio', 0.14, 0.006), ('expgrad_0.01', 'val', 'recall_overall', 0.436, 0.002),
    ('reweigh', 'val', 'fpr_ratio', 0.98, 0.006),      ('reweigh', 'test', 'fpr_ratio', 0.90, 0.006),
    ('reweigh', 'val', 'recall_overall', 0.527, 0.002),
    ('groupthr', 'val', 'fpr_ratio', 1.00, 0.006),     ('groupthr', 'test', 'fpr_ratio', 0.93, 0.006),
    ('groupthr', 'val', 'recall_overall', 0.536, 0.002),
]
est = ci.set_index(['model', 'split', 'metric']).estimate
for m, sp, k, exp, tol in EXPECTED:
    got = est[(m, sp, k)]
    print(f"{'ok   ' if abs(got - exp) <= tol else 'CHECK'} {m:13s} {sp:4s} {k:38s} got {got:.4f}  README {exp}")

ok    baseline      val  fpr_overall                            got 0.0500  README 0.05
ok    baseline      test fpr_overall                            got 0.0622  README 0.0622
ok    baseline      val  fpr_u50                                got 0.0368  README 0.0368
ok    baseline      val  fpr_50p                                got 0.1135  README 0.1135
ok    baseline      val  recall_overall                         got 0.5478  README 0.5478
ok    baseline      test recall_overall                         got 0.5893  README 0.5893
ok    baseline      val  fpr_ratio                              got 0.3247  README 0.32
ok    baseline      test fpr_ratio                              got 0.3249  README 0.33
ok    baseline      val  share_50p_of_wrongly_rejected_honest   got 0.3898  README 0.39
ok    expgrad_0.05  val  fpr_ratio                              got 0.4831  README 0.48
ok    expgrad_0.05  test fpr_ratio                              got 0.4671  README 0.47
ok    expgrad_0.05  va

## 6 · Headline table: FPR, FPR ratio and recall, with CIs

**What this does**
- Shows each headline metric as `estimate [lo, hi]`, one row per metric × model, with val and test side by side.
- `show()` is a small helper reused by the tables in 6c and 6d.

**How to read it**
- **FPR rows:** the harm to honest applicants in each group.
- **`fpr_ratio`:** closer to 1.0 is fairer.
- **Recall rows:** how much fraud is still caught.

**From the notes: L8 slide 23** (equalized odds: compare FPR *and* TPR across groups) and **L8 slide 9** (show the groups, because the overall number hides the gap).

In [ ]:
fmt = lambda r: f'{r.estimate:.3f} [{r.lo:.3f}, {r.hi:.3f}]'
show = lambda metrics: (ci[ci.metric.isin(metrics)].assign(ci95=lambda d: d.apply(fmt, axis=1))
                        .pivot_table(index=['metric', 'model'], columns='split', values='ci95', aggfunc='first')
                        .reindex(columns=['val', 'test']))
show(['fpr_u50', 'fpr_50p', 'fpr_ratio', 'recall_u50', 'recall_50p', 'recall_overall'])

split                                         val                  test
metric         model                                                   
fpr_50p        baseline      0.113 [0.109, 0.117]  0.143 [0.139, 0.147]
               expgrad_0.01  0.008 [0.007, 0.010]  0.013 [0.012, 0.015]
               expgrad_0.05  0.087 [0.084, 0.091]  0.109 [0.106, 0.112]
               groupthr      0.050 [0.047, 0.053]  0.067 [0.064, 0.070]
               reweigh       0.049 [0.046, 0.052]  0.066 [0.063, 0.068]
fpr_ratio      baseline      0.325 [0.310, 0.342]  0.325 [0.315, 0.336]
               expgrad_0.01  0.142 [0.121, 0.165]  0.187 [0.170, 0.206]
               expgrad_0.05  0.483 [0.459, 0.511]  0.467 [0.451, 0.484]
               groupthr      1.000 [0.924, 0.999]  0.932 [0.894, 0.973]
               reweigh       0.975 [0.913, 0.999]  0.900 [0.864, 0.939]
fpr_u50        baseline      0.037 [0.036, 0.038]  0.046 [0.046, 0.047]
               expgrad_0.01  0.059 [0.057, 0.060]  0.071 [0.070, 0.072]
               expgrad_0.05  0.042 [0.041, 0.043]  0.051 [0.050, 0.052]
               groupthr      0.050 [0.049, 0.051]  0.063 [0.061, 0.064]
               reweigh       0.050 [0.049, 0.052]  0.059 [0.058, 0.060]
recall_50p     baseline      0.723 [0.683, 0.765]  0.739 [0.713, 0.765]
               expgrad_0.01  0.209 [0.176, 0.251]  0.265 [0.237, 0.293]
               expgrad_0.05  0.668 [0.624, 0.711]  0.683 [0.653, 0.710]
               groupthr      0.557 [0.510, 0.603]  0.561 [0.531, 0.591]
               reweigh       0.523 [0.475, 0.570]  0.552 [0.522, 0.581]
recall_overall baseline      0.548 [0.521, 0.573]  0.589 [0.572, 0.607]
               expgrad_0.01  0.436 [0.409, 0.462]  0.485 [0.467, 0.504]
               expgrad_0.05  0.549 [0.522, 0.575]  0.582 [0.565, 0.600]
               groupthr      0.536 [0.508, 0.562]  0.565 [0.547, 0.582]
               reweigh       0.527 [0.500, 0.554]  0.558 [0.540, 0.576]
recall_u50     baseline      0.460 [0.428, 0.491]  0.511 [0.488, 0.533]
               expgrad_0.01  0.549 [0.516, 0.581]  0.600 [0.579, 0.623]
               expgrad_0.05  0.489 [0.457, 0.520]  0.530 [0.509, 0.553]
               groupthr      0.525 [0.492, 0.557]  0.566 [0.544, 0.589]
               reweigh       0.529 [0.496, 0.561]  0.561 [0.539, 0.585]

### 6b · Did each fix actually change anything? (paired change vs baseline)

**What this does**
- Shows the change vs baseline for each fix, with the paired CI from 4b: FPR ratio, overall recall, and recall and FPR for each age group.
- `excludes_0 = True` means the whole CI is on one side of zero, so the change is distinguishable from sampling noise.

**How to read it**
- `delta_fpr_ratio` > 0 means fairer than baseline.
- `delta_recall_overall` < 0 means the fix costs fraud detection.
- `delta_recall_50p` / `delta_recall_u50` show *who* loses or gains fraud detection. Example: ExpGrad 0.05's 50-plus recall 0.72 → 0.67 counts as real only if its `delta_recall_50p` row has `excludes_0 = True`.
- `delta_fpr_50p` < 0 means fewer honest 50-plus applicants are wrongly rejected; `delta_fpr_u50` > 0 means more honest under-50s are.
- Example question this answers: is reweighing's ~2pp recall loss real? That is "excludes_0" on its recall row.

**From the notes: L8 slide 30.** Fairness gains come with a cost when base rates differ, and this table says whether each gain and each cost is real.

In [ ]:
d = ci[ci.metric.str.startswith('delta_')].copy()
d['ci95'] = d.apply(fmt, axis=1)
d['excludes_0'] = (d.lo > 0) | (d.hi < 0)
d.pivot_table(index=['metric', 'model'], columns='split', values=['ci95', 'excludes_0'], aggfunc='first')

ci95  \
split                                                             test   
metric                           model                                   
delta_fpr_50p_vs_baseline        expgrad_0.01  -0.130 [-0.133, -0.126]   
                                 expgrad_0.05  -0.034 [-0.036, -0.032]   
                                 groupthr      -0.076 [-0.079, -0.073]   
                                 reweigh       -0.077 [-0.080, -0.075]   
delta_fpr_ratio_vs_baseline      expgrad_0.01  -0.138 [-0.162, -0.114]   
                                 expgrad_0.05     0.142 [0.132, 0.152]   
                                 groupthr         0.607 [0.576, 0.642]   
                                 reweigh          0.575 [0.544, 0.609]   
delta_fpr_u50_vs_baseline        expgrad_0.01     0.024 [0.023, 0.025]   
                                 expgrad_0.05     0.004 [0.004, 0.005]   
                                 groupthr         0.016 [0.015, 0.017]   
                                 reweigh          0.013 [0.012, 0.013]   
delta_recall_50p_vs_baseline     expgrad_0.01  -0.474 [-0.504, -0.444]   
                                 expgrad_0.05  -0.057 [-0.072, -0.042]   
                                 groupthr      -0.178 [-0.201, -0.154]   
                                 reweigh       -0.187 [-0.211, -0.163]   
delta_recall_overall_vs_baseline expgrad_0.01  -0.104 [-0.121, -0.088]   
                                 expgrad_0.05   -0.007 [-0.016, 0.002]   
                                 groupthr      -0.025 [-0.036, -0.013]   
                                 reweigh       -0.031 [-0.044, -0.019]   
delta_recall_u50_vs_baseline     expgrad_0.01     0.089 [0.077, 0.104]   
                                 expgrad_0.05     0.019 [0.009, 0.029]   
                                 groupthr         0.056 [0.046, 0.066]   
                                 reweigh          0.050 [0.038, 0.063]   

                                                                        \
split                                                              val   
metric                           model                                   
delta_fpr_50p_vs_baseline        expgrad_0.01  -0.105 [-0.109, -0.101]   
                                 expgrad_0.05  -0.026 [-0.028, -0.024]   
                                 groupthr      -0.063 [-0.067, -0.060]   
                                 reweigh       -0.064 [-0.068, -0.061]   
delta_fpr_ratio_vs_baseline      expgrad_0.01  -0.182 [-0.213, -0.153]   
                                 expgrad_0.05     0.158 [0.144, 0.175]   
                                 groupthr         0.675 [0.595, 0.680]   
                                 reweigh          0.651 [0.575, 0.681]   
delta_fpr_u50_vs_baseline        expgrad_0.01     0.022 [0.021, 0.023]   
                                 expgrad_0.05     0.005 [0.005, 0.006]   
                                 groupthr         0.013 [0.012, 0.014]   
                                 reweigh          0.013 [0.013, 0.014]   
delta_recall_50p_vs_baseline     expgrad_0.01  -0.515 [-0.561, -0.471]   
                                 expgrad_0.05  -0.055 [-0.076, -0.034]   
                                 groupthr      -0.166 [-0.201, -0.133]   
                                 reweigh       -0.200 [-0.235, -0.164]   
delta_recall_overall_vs_baseline expgrad_0.01  -0.112 [-0.136, -0.089]   
                                 expgrad_0.05    0.001 [-0.011, 0.013]   
                                 groupthr       -0.012 [-0.029, 0.004]   
                                 reweigh       -0.021 [-0.040, -0.002]   
delta_recall_u50_vs_baseline     expgrad_0.01     0.089 [0.071, 0.107]   
                                 expgrad_0.05     0.029 [0.014, 0.043]   
                                 groupthr         0.065 [0.051, 0.081]   
                                 reweigh          0.069 [0.051, 0.086]   

                                              excludes_0         
sp

### 6c · The "~39%" figure

**What this does**
- Shows, for each model, the share of **all wrongly rejected honest applicants** who are 50-plus, with a CI.
- Next to it, `share_50p_of_honest` shows the share 50-plus applicants *would* have if wrongful rejections were spread evenly (~17%). The gap between the two numbers is the unfairness, expressed as a share of the people harmed.
- This replaces the README's "~39% (approximate, Chloe's figures supersede)".

**From the notes: L8 slide 9.** Aggregate metrics hide what happens to a subgroup. Overall FPR is 5%, but 50-plus applicants carry a disproportionate share of the harm.

In [ ]:
show(['share_50p_of_honest', 'share_50p_of_wrongly_rejected_honest'])

split                                                               val  \
metric                               model                                
share_50p_of_honest                  baseline      0.172 [0.169, 0.174]   
                                     expgrad_0.01  0.172 [0.169, 0.174]   
                                     expgrad_0.05  0.172 [0.169, 0.174]   
                                     groupthr      0.172 [0.169, 0.174]   
                                     reweigh       0.172 [0.169, 0.174]   
share_50p_of_wrongly_rejected_honest baseline      0.390 [0.377, 0.401]   
                                     expgrad_0.01  0.029 [0.024, 0.033]   
                                     expgrad_0.05  0.300 [0.288, 0.312]   
                                     groupthr      0.172 [0.162, 0.182]   
                                     reweigh       0.168 [0.159, 0.178]   

split                                                              test  
metric                               model                               
share_50p_of_honest                  baseline      0.162 [0.161, 0.164]  
                                     expgrad_0.01  0.162 [0.161, 0.164]  
                                     expgrad_0.05  0.162 [0.161, 0.164]  
                                     groupthr      0.162 [0.161, 0.164]  
                                     reweigh       0.162 [0.161, 0.164]  
share_50p_of_wrongly_rejected_honest baseline      0.373 [0.365, 0.381]  
                                     expgrad_0.01  0.035 [0.032, 0.038]  
                                     expgrad_0.05  0.293 [0.285, 0.301]  
                                     groupthr      0.172 [0.165, 0.178]  
                                     reweigh       0.177 [0.171, 0.184]

### 6d · Bonus: precision and flag rate by group

**What this does**
- Shows precision (of the applicants flagged, the share who really are fraud) and flag rate (the share of the group flagged) for each age group, with CIs.
- Read the **groupthr** rows. The README quotes precision of 9% (under-50) vs 21% (50-plus). The other models are shown only for comparison.

**Why groupthr in particular:** group thresholds force equal FPR by giving each group its own cutoff. The 50-plus fraud rate is higher (2.34% vs 0.83%), so equal FPR forces **unequal precision**: a flagged under-50 is much less likely to really be fraud.

**From the notes: L8 slide 30.** With different base rates you cannot have equal precision and balanced error rates at the same time. This table is that result showing up in our own data. L8 slide 22 defines PPV (precision) and selection rate (flag rate).

In [ ]:
show(['precision_u50', 'precision_50p', 'flag_rate_u50', 'flag_rate_50p'])

split                                        val                  test
metric        model                                                   
flag_rate_50p baseline      0.127 [0.123, 0.131]  0.161 [0.157, 0.164]
              expgrad_0.01  0.013 [0.011, 0.014]  0.021 [0.019, 0.022]
              expgrad_0.05  0.101 [0.096, 0.104]  0.126 [0.122, 0.129]
              groupthr      0.062 [0.058, 0.065]  0.082 [0.079, 0.084]
              reweigh       0.060 [0.056, 0.063]  0.080 [0.077, 0.082]
flag_rate_u50 baseline      0.041 [0.040, 0.042]  0.052 [0.051, 0.053]
              expgrad_0.01  0.063 [0.062, 0.065]  0.077 [0.075, 0.078]
              expgrad_0.05  0.047 [0.045, 0.048]  0.056 [0.055, 0.057]
              groupthr      0.055 [0.053, 0.056]  0.068 [0.067, 0.069]
              reweigh       0.055 [0.053, 0.056]  0.065 [0.064, 0.066]
precision_50p baseline      0.129 [0.116, 0.141]  0.135 [0.125, 0.144]
              expgrad_0.01  0.367 [0.309, 0.425]  0.376 [0.339, 0.414]
              expgrad_0.05  0.151 [0.135, 0.165]  0.159 [0.148, 0.170]
              groupthr      0.205 [0.183, 0.226]  0.202 [0.187, 0.215]
              reweigh       0.199 [0.175, 0.220]  0.202 [0.187, 0.216]
precision_u50 baseline      0.107 [0.097, 0.117]  0.109 [0.103, 0.116]
              expgrad_0.01  0.083 [0.076, 0.089]  0.086 [0.082, 0.092]
              expgrad_0.05  0.100 [0.091, 0.108]  0.104 [0.098, 0.111]
              groupthr      0.092 [0.084, 0.099]  0.092 [0.086, 0.097]
              reweigh       0.092 [0.084, 0.099]  0.096 [0.090, 0.102]

## 7 · Save the CSV

**What this does**
- Writes the full table to a **new** file, `models/bootstrap_cis.csv`, with columns `model, split, metric, estimate, lo, hi`. Re-running overwrites only this file.
- Writes nothing to `preds/` and never opens `proxy_set.json`; both are frozen.


In [ ]:
OUT = f'{MODELS_DIR}/bootstrap_cis.csv'
ci[['model', 'split', 'metric', 'estimate', 'lo', 'hi']].to_csv(OUT, index=False)
print('saved', OUT, len(ci), 'rows')
print(f'\nbootstrap CIs are in CS3268_Project/models/bootstrap_cis.csv '
      f'(columns model, split, metric, estimate, lo, hi; {len(ci)} rows; notebook notebooks/08_chloe_bootstrap_ci.ipynb).')

saved /content/drive/MyDrive/CS3268_Project/models/bootstrap_cis.csv 178 rows

bootstrap CIs are in CS3268_Project/models/bootstrap_cis.csv (columns model, split, metric, estimate, lo, hi; 178 rows; notebook notebooks/08_chloe_bootstrap_ci.ipynb).


## 8 · Text for the report (Method + caveats)

> **Uncertainty.** We report 95% percentile bootstrap confidence intervals (1000 resamples, seed 42) for per-group FPR and recall, the FPR ratio (lower/higher), overall recall, and the share of wrongly rejected honest applicants who are 50-plus, separately on validation (month 5, n = 119,323) and test (months 6–7, n = 205,011). Resampling is over applicants (rows), using each model's **saved decisions**: cutoffs are held fixed and not re-estimated within resamples, so intervals reflect sampling variation in the evaluation set only, not uncertainty from training or threshold selection. Rows are treated as independent, and test months 6 and 7 are pooled. Changes relative to the baseline use a **paired** bootstrap: the same resampled rows are used for the baseline and each fix, and the difference is computed within each resample.

Caveats to state:
- All cutoffs were set on validation (month 5), and ExpGrad's bound (0.05) was also selected there by the pre-registered rule, so validation results are in-sample for these choices. Test is the cleaner estimate. In practice ExpGrad 0.05 changed little between splits (FPR ratio 0.48 val, 0.47 test).
- Group thresholds reach FPR parity on validation **by construction** (ratio 1.000). Because the ratio cannot exceed 1, its bootstrap interval on validation lies below the estimate ([0.924, 0.999]) and should not be read as evidence of fairness. On test the ratio is 0.93 [0.89, 0.97]: close to parity, but the interval excludes 1.
- Overall test FPR is about 6% for every model, against the 5% target on validation, and every group's FPR is higher on test than on validation. This is consistent with drift across months (L6 s14; L7 s10–12). The CIs describe these months only and do not cover future drift.
- CIs are not corrected for multiple comparisons across models and metrics. Some results only just exclude zero, for example reweighing's validation recall change (−0.021 [−0.040, −0.002]), or exclude zero on one split only (group thresholds' recall change: test −0.025 [−0.036, −0.013], validation −0.012 [−0.029, 0.004]). Treat these with caution. Some changes are clearly non-zero but very small (ExpGrad 0.05's under-50 FPR: +0.004).
- Treating applicants as independent ignores repeat applicants and month-to-month variation (L7 s9), so the intervals may be somewhat too narrow.